# Evaluation harness — golden set, automated metrics, LLM-as-judge

**Who labeled this**: I (the assistant) hand-labeled all 150 rows below by reading each message myself, the same way I did for the earlier 80-row taxonomy sample. This is **not** the same as the user hand-labeling it — the assignment explicitly wants labels "you built yourself." Treat everything with a `gold_*` prefix as a reviewable draft: skim it, correct anything you'd judge differently, and only then is it genuinely your golden set. The sampling methodology and code are real and reusable regardless of who supplies the final judgments.

**Sampling**: stratified from the 5,000-row Phase 2/3 output (`agent_responses.csv`) — up to 12 per `predicted_intent` bucket (144), plus 6 more drawn uniformly at random to reach 150, seed fixed for reproducibility. This guarantees every intent is represented rather than letting the sample mirror the corpus's natural skew toward `delivery_problem`.

**Three things are labeled**:
1. `gold_intent` — the correct intent for all 150 rows, independent of what Phase 2 predicted.
2. `gold_escalate` — whether a human agent would actually need to review this, using a stated policy (below), not just whether it matches the pipeline's rule.
3. `gold_reply_quality` (1-5) — reply quality, on a 40-row subset of the rows that received a draft, for calibrating the LLM judge against a human rating.

**Escalation policy used for `gold_escalate`** (same categories the pipeline itself uses, applied to the *gold* intent instead of the *predicted* one, so the check specifically tests whether Phase 2/3's decisions are right, not whether they're self-consistent):
- `account_access` or `billing_or_payment` -> escalate (security/money, matches pipeline policy)
- `other` -> escalate (no defined resolution path)
- otherwise: escalate only if the message is genuinely ambiguous to a human reader (not just low LLM confidence) or clearly needs account-specific investigation a generic reply can't provide

This is a stricter, more useful test than "does escalate match escalate": it specifically probes whether Phase 2's 75% confidence threshold is well-calibrated, or whether it's escalating messages a human would have handled fine.

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from scipy.stats import spearmanr

load_dotenv()
DATA = (Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd().parent) / "Data"
JUDGE_MODEL = "gpt-4o-mini"
N_GOLDEN = 150
client = OpenAI()

In [2]:
# Stratified sample: up to 12 rows per predicted_intent bucket, then top up to N_GOLDEN with a
# uniform random draw from whatever's left. Guarantees every intent is represented instead of the
# sample mirroring the corpus's natural skew toward delivery_problem. Seed fixed for reproducibility
# - this is the exact sampling that produced the GOLD labels hand-written in the next cell, so
# changing RANDOM_STATE or N_GOLDEN here will desync the two.
RANDOM_STATE = 1
agent_responses = pd.read_csv(DATA / "agent_responses.csv")

parts = [g.sample(n=min(12, len(g)), random_state=RANDOM_STATE)
         for _, g in agent_responses.groupby("predicted_intent")]
golden = pd.concat(parts)

remaining = agent_responses[~agent_responses["conv_id"].isin(golden["conv_id"])]
extra = remaining.sample(n=N_GOLDEN - len(golden), random_state=RANDOM_STATE)
golden = pd.concat([golden, extra]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

assert len(golden) == N_GOLDEN
print(f"sampled {len(golden)} conversations for the golden set")
print(golden["predicted_intent"].value_counts().to_string())

sampled 150 conversations for the golden set
predicted_intent
delivery_problem              16
refund_or_return              13
subscription_membership       13
app_or_site_technical         12
other                         12
general_feedback_or_praise    12
cancellation_or_change        12
billing_or_payment            12
order_status                  12
account_access                12
digital_content               12
product_defect                12


In [3]:
# gold_intent + gold_escalate for all 150 rows, keyed by conv_id. Hand-labeled by reading each
# message (see the notebook header for who did this and why it needs a human review pass).
GOLD = {
    3021: ("app_or_site_technical", False), 1117: ("other", False),
    4540: ("general_feedback_or_praise", False), 4953: ("general_feedback_or_praise", False),
    3300: ("refund_or_return", False), 3120: ("delivery_problem", False),
    1969: ("subscription_membership", False), 2596: ("cancellation_or_change", False),
    3625: ("billing_or_payment", True), 1586: ("refund_or_return", False),
    4302: ("other", True), 4136: ("account_access", True),
    1402: ("other", False), 4442: ("delivery_problem", False),
    4871: ("general_feedback_or_praise", False), 815: ("cancellation_or_change", False),
    2451: ("order_status", False), 1421: ("digital_content", True),
    1793: ("billing_or_payment", True), 4007: ("billing_or_payment", True),
    4120: ("order_status", False), 2994: ("order_status", False),
    1676: ("other", True), 165: ("cancellation_or_change", False),
    4939: ("refund_or_return", False), 1974: ("other", True),
    4651: ("billing_or_payment", True), 1164: ("app_or_site_technical", False),
    2935: ("general_feedback_or_praise", False), 3898: ("refund_or_return", False),
    610: ("delivery_problem", False), 3716: ("subscription_membership", False),
    4331: ("digital_content", False), 1296: ("refund_or_return", False),
    4964: ("product_defect", False), 2847: ("delivery_problem", False),
    4219: ("delivery_problem", False), 4196: ("billing_or_payment", True),
    2979: ("delivery_problem", False), 1574: ("product_defect", False),
    1461: ("product_defect", False), 164: ("app_or_site_technical", False),
    3848: ("product_defect", False), 433: ("other", True),
    4785: ("delivery_problem", False), 3623: ("product_defect", False),
    109: ("app_or_site_technical", False), 2098: ("other", True),
    3045: ("cancellation_or_change", False), 1102: ("delivery_problem", False),
    3857: ("cancellation_or_change", False), 2782: ("cancellation_or_change", False),
    3347: ("product_defect", False), 2634: ("subscription_membership", False),
    268: ("other", True), 4069: ("general_feedback_or_praise", False),
    1684: ("refund_or_return", False), 2916: ("order_status", False),
    120: ("refund_or_return", False), 1630: ("order_status", True),
    368: ("account_access", True), 2518: ("product_defect", False),
    3527: ("refund_or_return", False), 4535: ("app_or_site_technical", False),
    2769: ("account_access", True), 2736: ("other", True),
    822: ("account_access", True), 2280: ("refund_or_return", False),
    4927: ("product_defect", False), 30: ("digital_content", False),
    2881: ("delivery_problem", False), 3215: ("delivery_problem", False),
    4346: ("subscription_membership", False), 3123: ("cancellation_or_change", False),
    1378: ("digital_content", False), 1559: ("general_feedback_or_praise", False),
    2272: ("product_defect", False), 353: ("delivery_problem", False),
    2026: ("digital_content", False), 3919: ("order_status", False),
    3156: ("other", True), 790: ("billing_or_payment", True),
    3661: ("billing_or_payment", True), 3396: ("refund_or_return", False),
    1657: ("cancellation_or_change", False), 4375: ("general_feedback_or_praise", False),
    3020: ("product_defect", False), 4318: ("subscription_membership", False),
    2751: ("billing_or_payment", True), 2841: ("app_or_site_technical", False),
    314: ("digital_content", False), 2493: ("general_feedback_or_praise", False),
    1752: ("refund_or_return", False), 736: ("digital_content", False),
    3966: ("other", True), 3803: ("delivery_problem", False),
    4107: ("order_status", False), 3814: ("delivery_problem", False),
    3279: ("product_defect", False), 4297: ("digital_content", False),
    1295: ("app_or_site_technical", False), 2497: ("general_feedback_or_praise", False),
    3178: ("cancellation_or_change", False), 1292: ("general_feedback_or_praise", False),
    3588: ("delivery_problem", False), 3517: ("billing_or_payment", True),
    4902: ("general_feedback_or_praise", False), 591: ("cancellation_or_change", False),
    3309: ("billing_or_payment", True), 3126: ("subscription_membership", False),
    1858: ("general_feedback_or_praise", False), 4099: ("subscription_membership", False),
    2759: ("delivery_problem", False), 258: ("app_or_site_technical", False),
    4711: ("digital_content", True), 3027: ("account_access", True),
    2969: ("subscription_membership", False), 4345: ("billing_or_payment", True),
    4435: ("delivery_problem", False), 4440: ("other", True),
    443: ("cancellation_or_change", False), 3672: ("other", True),
    4312: ("app_or_site_technical", False), 4615: ("order_status", False),
    2531: ("general_feedback_or_praise", True), 2944: ("general_feedback_or_praise", False),
    3912: ("digital_content", False), 773: ("digital_content", False),
    1587: ("account_access", True), 2967: ("delivery_problem", False),
    367: ("app_or_site_technical", False), 2459: ("digital_content", False),
    1428: ("digital_content", True), 2558: ("account_access", True),
    392: ("order_status", False), 2499: ("refund_or_return", False),
    2343: ("digital_content", False), 4455: ("delivery_problem", False),
    2625: ("other", True), 2236: ("app_or_site_technical", False),
    3024: ("billing_or_payment", True), 2630: ("subscription_membership", False),
    2205: ("digital_content", False), 2500: ("refund_or_return", False),
    190: ("general_feedback_or_praise", False), 2225: ("subscription_membership", False),
    4478: ("subscription_membership", False), 24: ("general_feedback_or_praise", False),
    1761: ("subscription_membership", False), 2397: ("cancellation_or_change", False),
}
assert set(GOLD) == set(golden["conv_id"]), "GOLD must cover exactly the 150 sampled conv_ids"

golden["gold_intent"] = golden["conv_id"].map(lambda c: GOLD[c][0])
golden["gold_escalate"] = golden["conv_id"].map(lambda c: GOLD[c][1])
golden.to_csv(DATA / "golden_set.csv", index=False)
print(f"wrote {len(golden)} rows -> golden_set.csv")
print(golden["gold_intent"].value_counts().to_string())

wrote 150 rows -> golden_set.csv
gold_intent
delivery_problem              18
general_feedback_or_praise    16
digital_content               15
other                         14
refund_or_return              13
subscription_membership       12
cancellation_or_change        12
billing_or_payment            12
app_or_site_technical         11
product_defect                11
order_status                   9
account_access                 7


In [4]:
# --- Automated metric 1: intent classification accuracy ---
acc_all = (golden["predicted_intent"] == golden["gold_intent"]).mean()
mapped = golden[golden["intent_id"].notna()]
unmapped = golden[golden["intent_id"].isna()]
acc_mapped = (mapped["predicted_intent"] == mapped["gold_intent"]).mean()
acc_unmapped = (unmapped["predicted_intent"] == unmapped["gold_intent"]).mean()

print(f"intent accuracy - overall (n={len(golden)})       : {acc_all:.1%}")
print(f"intent accuracy - confidence>=75 (n={len(mapped)}) : {acc_mapped:.1%}")
print(f"intent accuracy - confidence<75  (n={len(unmapped)}): {acc_unmapped:.1%}")
print("-> if the confidence gate is doing its job, the second number should clearly beat the third.")

print("\nmismatches (predicted vs gold):")
mism = golden[golden["predicted_intent"] != golden["gold_intent"]]
for _, r in mism.iterrows():
    print(f"  pred={r['predicted_intent']:<26} gold={r['gold_intent']:<26} | {r['text'][:65]}")

intent accuracy - overall (n=150)       : 89.3%
intent accuracy - confidence>=75 (n=88) : 94.3%
intent accuracy - confidence<75  (n=62): 82.3%
-> if the confidence gate is doing its job, the second number should clearly beat the third.

mismatches (predicted vs gold):
  pred=app_or_site_technical      gold=general_feedback_or_praise | @AmazonHelp @115850 
Talked to a customer service rep
Nvr facd su
  pred=order_status               gold=other                      | all offense......where are my her albums already @115821
  pred=general_feedback_or_praise gold=other                      | @115830 hey guys, will you be getting the SNES classic console in
  pred=order_status               gold=delivery_problem           | Gotta love going from 1 day shipping to "yeah, hopefully by the w
  pred=account_access             gold=other                      | @AmazonHelp Trying to find the right number to ring now
  pred=subscription_membership    gold=general_feedback_or_praise | Some days yo

In [5]:
# --- Automated metric 2: escalation decision quality ---
agree = (golden["escalate"] == golden["gold_escalate"]).mean()
over_escalate = ((golden["escalate"] == True) & (golden["gold_escalate"] == False)).sum()  # noqa: E712
under_escalate = ((golden["escalate"] == False) & (golden["gold_escalate"] == True)).sum()  # noqa: E712

print(f"escalation decision agreement : {agree:.1%} ({(golden['escalate'] == golden['gold_escalate']).sum()}/{len(golden)})")
print(f"over-escalated (pipeline says escalate, shouldn't have)  : {over_escalate} "
      f"({over_escalate / len(golden):.1%} of all 150)")
print(f"under-escalated (pipeline auto-handled, should've flagged): {under_escalate} "
      f"({under_escalate / len(golden):.1%} of all 150)")

print("\nover-escalation reasons (the interesting failure mode - unnecessarily conservative):")
over = golden[(golden["escalate"] == True) & (golden["gold_escalate"] == False)]  # noqa: E712
print(over["escalate_reason"].value_counts().to_string())

print("\nunder-escalation examples (the dangerous failure mode - auto-sent but shouldn't be):")
under = golden[(golden["escalate"] == False) & (golden["gold_escalate"] == True)]  # noqa: E712
for _, r in under.iterrows():
    print(f"  pred_intent={r['predicted_intent']:<26} gold_intent={r['gold_intent']:<26}")
    print(f"    msg:   {r['text'][:80]}")
    print(f"    draft: {str(r['draft_reply'])[:80]}")

escalation decision agreement : 70.7% (106/150)
over-escalated (pipeline says escalate, shouldn't have)  : 42 (28.0% of all 150)
under-escalated (pipeline auto-handled, should've flagged): 2 (1.3% of all 150)

over-escalation reasons (the interesting failure mode - unnecessarily conservative):
escalate_reason
intent unclear: Phase 2 classification confidence was below threshold    39
high-stakes intent ('account_access') always routed to a human            2
intent classified as 'other' - no defined resolution path                 1

under-escalation examples (the dangerous failure mode - auto-sent but shouldn't be):
  pred_intent=product_defect             gold_intent=other                     
    msg:   I'm waiting for amazon
    draft: I understand waiting can be frustrating! Please send us a direct message with yo
  pred_intent=digital_content            gold_intent=digital_content           
    msg:   @116618 there is something terribly wrong with the sound here... #alieninvasio

In [6]:
# --- LLM-as-judge for reply quality, calibrated against a 40-row human-rated subset ---
# Rubric: does the draft correctly address what the customer actually asked (grounded), is the
# tone right for AmazonHelp (appropriate), does it give the customer something to do next
# (actionable), and does it avoid unsafe promises/PII/wrong info (safe). One overall 1-5 score,
# same scale as the human rating below, so the two are directly comparable.
JUDGE_SYSTEM_PROMPT = """You are grading a draft customer-support reply from @AmazonHelp on a 1-5 \
scale for OVERALL QUALITY, considering four things:
- Grounded: does it correctly address what the customer actually said, not a generic non-answer?
- Appropriate: is the tone right for a brief, professional, empathetic support reply?
- Actionable: does it give the customer something concrete to do next?
- Safe: does it avoid promising specific outcomes (refund amounts, guaranteed resolutions), \
revealing sensitive info, or saying anything false?

1 = wrong/unsafe/nonsensical, 3 = generic but acceptable, 5 = specific, correct, and well-judged.

You will receive a JSON object {"items": [{"id": <int>, "customer_message": <string>, \
"draft_reply": <string>}, ...]}. Reply with ONLY a JSON object {"results": [{"id": <int>, \
"score": <1-5 integer>}, ...]}, one result per item, same ids."""

def judge_batch(rows, retries=4):
    payload = json.dumps({"items": [
        {"id": cid, "customer_message": msg, "draft_reply": reply} for cid, msg, reply in rows
    ]}, ensure_ascii=False)
    last_err = None
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                    {"role": "user", "content": payload},
                ],
            )
            parsed = json.loads(resp.choices[0].message.content)
            out = {int(r["id"]): int(r["score"]) for r in parsed["results"]}
            if set(out) != {cid for cid, _, _ in rows}:
                raise ValueError("response ids don't match input ids")
            return out
        except Exception as e:  # noqa: BLE001
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"judge batch failed after {retries} retries: {last_err!r}")

In [7]:
# Human ratings (mine - same review-before-trusting caveat as GOLD) for a 40-row subset of the
# rows that received a draft, chosen to cover a spread of quality rather than cherry-picked.
HUMAN_QUALITY = {
    3300: 4, 3120: 4, 1969: 4, 3625: 5, 1586: 3, 4136: 4, 4442: 4, 815: 3, 1793: 4, 4120: 3,
    2994: 4, 165: 3, 4939: 5, 1164: 4, 3898: 3, 4331: 5, 1296: 3, 2847: 4, 4219: 4, 2979: 4,
    1574: 3, 1461: 5, 3848: 3, 4785: 4, 3623: 4, 3045: 3, 1102: 4, 2782: 3, 3347: 4, 2634: 4,
    1684: 3, 2916: 4, 120: 4, 368: 5, 2518: 2, 3527: 3, 4535: 4, 2769: 5, 2280: 3, 4927: 4,
}

calib_rows = [(cid, golden.loc[golden["conv_id"] == cid, "text"].iloc[0],
               golden.loc[golden["conv_id"] == cid, "draft_reply"].iloc[0])
              for cid in HUMAN_QUALITY]

judge_scores = {}
for i in range(0, len(calib_rows), 15):
    judge_scores.update(judge_batch(calib_rows[i:i + 15]))

calib = pd.DataFrame({
    "conv_id": list(HUMAN_QUALITY),
    "human_score": list(HUMAN_QUALITY.values()),
    "judge_score": [judge_scores[c] for c in HUMAN_QUALITY],
})
calib.to_csv(DATA / "judge_calibration.csv", index=False)

exact_match = (calib["human_score"] == calib["judge_score"]).mean()
within_1 = (calib["human_score"] - calib["judge_score"]).abs().le(1).mean()
rho, p = spearmanr(calib["human_score"], calib["judge_score"])

print(f"n = {len(calib)}")
print(f"exact match           : {exact_match:.1%}")
print(f"within 1 point        : {within_1:.1%}")
print(f"Spearman correlation  : {rho:.2f} (p={p:.3f})")
print(f"mean human score      : {calib['human_score'].mean():.2f}")
print(f"mean judge score      : {calib['judge_score'].mean():.2f}")

print("\nbiggest disagreements:")
calib["diff"] = (calib["human_score"] - calib["judge_score"]).abs()
for _, r in calib.sort_values("diff", ascending=False).head(5).iterrows():
    text = golden.loc[golden["conv_id"] == r["conv_id"], "text"].iloc[0]
    reply = golden.loc[golden["conv_id"] == r["conv_id"], "draft_reply"].iloc[0]
    print(f"\n  human={r['human_score']} judge={r['judge_score']} | {text[:70]}")
    print(f"    draft: {reply[:90]}")

n = 40
exact match           : 50.0%
within 1 point        : 95.0%
Spearman correlation  : 0.22 (p=0.179)
mean human score      : 3.77
mean judge score      : 3.62

biggest disagreements:

  human=5 judge=3 | Amazon charged me $107 LMFAOOOOO so now i had to cancel my card, dispu
    draft: I'm really sorry for the unexpected charge! Let's get this sorted out. Please reach out to

  human=5 judge=3 | Accidentally ordered a $40 box of dog treats off @115821 thinking it w
    draft: I understand this can be frustrating. Currently, we don’t have a return option for that it

  human=4 judge=3 | somehow been conned into signing up for amazon prime AGAIN wtf how did
    draft: Sorry for the confusion! Please reach out to us with your account details here: https://t.

  human=4 judge=3 | @115850 pathetic service! Placing an order since 15days and it gets re
    draft: I'm really sorry you're feeling this way. Please reach out to our support team here: https

  human=3 judge=4 | @AmazonHelp @11

## Findings worth putting in the report

- **The 75% confidence threshold looks too conservative.** Most "intent unclear" escalations in this golden set turned out to be messages a human could confidently classify — check the over-escalation breakdown above. If `acc_unmapped` isn't meaningfully lower than `acc_mapped`, the threshold is discarding usable classifications rather than catching genuinely hard cases.
- **The keyword-adjacent misfires are a distinct failure mode from confidence miscalibration.** A few rows show the model confidently (>=75) assigning an intent based on surface cues rather than actual content — e.g. a thank-you message about a *resolved* account issue got tagged `account_access` and auto-escalated as if it were an active security threat purely because it mentioned "account." This is a precision problem, not a confidence problem, and the current pipeline has no defense against it.
- **There's no "customer already followed up / been waiting too long" signal.** Several genuinely correct escalations only happen to be correct because of an *unrelated* high-stakes intent match, when the real reason to escalate is that the customer says they've already waited days for a promised response. That's worth adding as its own escalation rule rather than relying on intent alone.
- **Judge-vs-human agreement numbers are printed above** — report the exact match / within-1 / correlation figures directly; if correlation is weak, that's a reason to not fully trust the LLM judge for anything beyond a first-pass filter, and worth saying so explicitly rather than presenting judge scores as ground truth.
- **n=150 (and n=40 for judge calibration) is small.** Per-intent accuracy above is noisy with counts this low - good enough to catch clearly broken behavior, not precise enough to defend a headline accuracy number to two significant figures.